---
author: Logan Foster
format:
  html:
    code-tools: true
    code-fold: true 
---

## Problem 1: Metropolis Hasting with Independent Sampler


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def gauss(x, mu, omega):

    if len(x.shape) == 1:
        x = x.reshape((1, x.shape[0]))

    return (0.5 * omega / np.pi) * np.exp(-0.5 * omega * np.sum((x - mu)**2, axis = -1))

def grad_gauss(x, mu, omega):
    g = gauss(x, mu, omega)
    return (-0.5 * omega**2 / np.pi) * (g.reshape((g.shape[0], 1)).repeat(2, axis = 1) * (x - mu))


def phi(x):
    if len(x.shape) == 1 or x.shape[0] == 0:
        return np.column_stack((x[0], x[1] + 0.25 * np.sin(4.0 * np.pi * x[0])))
    else:
        return np.column_stack((x[:, 0], x[:, 1] + 0.25 * np.sin(4.0 * np.pi * x[:, 0])))

# This computes the derivative mapping transpose, times a vector v, that is: dphi(x)^T * v
def dphi_transpose_dot_v(x, v):

    if len(x.shape) == 1:
        x = x.reshape((1, x.shape[0]))


    return np.column_stack(
        (
            v[:, 0] + v[:, 1] * np.pi * np.cos(4.0 * np.pi * x[:, 0]),      # v_1 * 1 + v_2 * pi cos (4 pi x_1)
            v[:, 1]                                                         # v_1 * 0 + v_2 * 1
        )
    ) 

# Parameters for the pdf
mu1 = np.array([1.0, 1.0])
mu2 = np.array([-1.0, -1.0])
omega1 = 9.0
omega2 = 9.0
lambda1 = 0.4
lambda2 = 0.6

# For taking logs near zero
epsilon = 1.0e-8

def U(x):
    e1 = lambda1 * gauss(x, mu1, omega1)
    e2 = lambda2 * gauss(phi(x), mu2, omega2)

    return -np.log(e1 + e2 + epsilon)

Let's setup the MH sampler.


In [ ]:
## proposal dist is N(x, mu, sigma^2 * I), so a standard gaussian. We'll need to draw samples from this standard normal (simple numpy) and evaluate the likelihood of past samples




class MH_indp_normal():
    def __init__(self, N = 10000, proposal_var = 1, proposal_mean = np.array([0,0])):
        self.N = N
        self.proposal_var = proposal_var
        self.proposal_mean = proposal_mean
    
    # @staticmethod
    def proposal(self, x, mean = None, var = None):
        if mean is None:
            mean = self.proposal_mean
        if var is None:
            var = self.proposal_var
        
        return gauss(x, mean, 1 / var)

    def generate_proposal(self, mean = None, var = None):
        
        if mean is None:
            mean = self.proposal_mean
        if var is None:
            var = self.proposal_var
        
        return np.random.normal(mean, var * np.array([1,1]))

    def target(self, x): # not normalized!!
        return np.exp(-U(x))

    def sample(self):
        num_samples = self.N
        self.samples = np.zeros((num_samples,2))
        samples = self.samples
        self.accepted = np.zeros(num_samples - 1)

        for k in range(1, num_samples):
            x_last = samples[k-1]

            y = self.generate_proposal()

            u = np.random.uniform(0,1)

            A = self.target(y) * self.proposal(x_last) / (self.target(x_last) * self.proposal(y))

            if u < A:
                samples[k] = y
                self.accepted[k - 1] = 1
            else:
                samples[k] = x_last

### Part A: First Sampling Batch ($\sigma = 1$)
Running our MH independent sampler with $\sigma = 1$.

In [ ]:
N = 10_000
sampler = MH_indp_normal(proposal_var = 1)
sampler.sample()
accept_rate = 100 * sampler.accepted.sum() / N
print(f"Acceptance rate: {accept_rate: .2f}%")

### Part B: Plotting

In [ ]:
# Make grid for plotting
xcoord = np.linspace(-3.0, 3.0, 201)
ycoord = np.linspace(-3.0, 3.0, 201)
xx, yy = np.meshgrid(xcoord, ycoord)

# reshape xx,yy coordinates into N x 2 tensor
coords = np.column_stack((xx.flatten(), yy.flatten()))

# compute (unnormalized) density function
dens = np.exp(-U(coords))

# reshape back to the xx,yy grid
dens = dens.reshape(xx.shape)

# Contour plot
levels = np.linspace(np.max(dens) / 100, np.max(dens), 99)
plt.figure(figsize = (10, 10))
ax = plt.axes()
ax.set_facecolor("black")
plt.contourf(xx, yy, dens, levels = levels, cmap = "gist_heat")

samples = sampler.samples
plt.scatter(samples[:, 0], samples[:,1], s = 1, c = 'white')

plt.title(r"Scattered Samples Generated with $\sigma^2 = 1$")
plt.grid(color='white', linestyle = '--')

plt.axis('equal')
plt.show()

### Part C: Experiments with different $\sigma^2$

In [ ]:
# Make grid for plotting (shared across all subplots)
xcoord = np.linspace(-3.0, 3.0, 201)
ycoord = np.linspace(-3.0, 3.0, 201)
xx, yy = np.meshgrid(xcoord, ycoord)
coords = np.column_stack((xx.flatten(), yy.flatten()))
dens = np.exp(-U(coords)).reshape(xx.shape)
levels = np.linspace(np.max(dens) / 100, np.max(dens), 99)

fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)

for ax, var in zip(axes.flatten(), [0.1, 0.5, 1, 2]):
    sampler = MH_indp_normal(proposal_var = var)
    sampler.sample()
    accept_rate = 100 * sampler.accepted.sum() / N
    print(f"Proposal Variance: {var}    Acceptance rate: {accept_rate: .2f}%")

    ax.set_facecolor("black")
    ax.contourf(xx, yy, dens, levels = levels, cmap = "gist_heat")

    samples = sampler.samples
    ax.scatter(samples[:, 0], samples[:, 1], s = 1, c = 'white')

    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title(f"Scattered Samples Generated with $\\sigma^2 = {var}$ \n Acceptance rate: {accept_rate: .2f}%")
    ax.grid(color='white', linestyle = '--')
    ax.set_aspect('equal')

plt.show()

This is interesting. When are variance is too small ( ~ $\sigma^2 < 1$), the acceptance probability is very high, but the samples don't resemble the target distribution, they look more like the proposal distribution. As the variance gets larger, the acceptance probability falls off, bu the samples look a lot more like the target dist.

## Problem 2: MH Sampling with Dependent Proposal Dist

In [ ]:
class MH_dep_normal():
    def __init__(self, N = 10000, proposal_var = 1, proposal_mean = np.array([0,0])):
        self.N = N
        self.proposal_var = proposal_var
        self.proposal_mean = proposal_mean
    
    # @staticmethod
    def proposal(self, x, mean):

        return gauss(x, mean, 1 / var)

    def generate_proposal(self, mean):
        return np.random.normal(mean, self.proposal_var * np.array([1,1]))

    def target(self, x): # not normalized!!
        return np.exp(-U(x))

    def sample(self):
        num_samples = self.N
        self.samples = np.zeros((num_samples,2))
        samples = self.samples
        self.accepted = np.zeros(num_samples - 1)

        for k in range(1, num_samples):
            x_last = samples[k-1]

            y = self.generate_proposal(x_last)

            u = np.random.uniform(0,1)

            A = self.target(y) * self.proposal(x_last, y) / (self.target(x_last) * self.proposal(y, x_last))

            if u < A:
                samples[k] = y
                self.accepted[k - 1] = 1
            else:
                samples[k] = x_last

### Part A: Acceptance Rate

In [ ]:
N = 10_000
sampler = MH_dep_normal(proposal_var = 1)
sampler.sample()
accept_rate = 100 * sampler.accepted.sum() / N
print(f"Acceptance rate: {accept_rate: .2f}%")

### Part B: Plotting

In [ ]:
# Make grid for plotting
xcoord = np.linspace(-3.0, 3.0, 201)
ycoord = np.linspace(-3.0, 3.0, 201)
xx, yy = np.meshgrid(xcoord, ycoord)

# reshape xx,yy coordinates into N x 2 tensor
coords = np.column_stack((xx.flatten(), yy.flatten()))

# compute (unnormalized) density function
dens = np.exp(-U(coords))

# reshape back to the xx,yy grid
dens = dens.reshape(xx.shape)

# Contour plot
levels = np.linspace(np.max(dens) / 100, np.max(dens), 99)
plt.figure(figsize = (10, 10))
ax = plt.axes()
ax.set_facecolor("black")
plt.contourf(xx, yy, dens, levels = levels, cmap = "gist_heat")

samples = sampler.samples
plt.scatter(samples[:, 0], samples[:,1], s = 1, c = 'white')

plt.title("Scattered Samples Generated Dependently \n" + r"$\sigma^2" + "= 1$  Acceptance rate: " + f"{accept_rate: .2f}%")
plt.grid(color='white', linestyle = '--')

plt.axis('equal')
plt.show()

Right off the bat, it seems like our sampling rate is significantly better!


### Part C: Experiments with different $\sigma^2$

In [ ]:
# Make grid for plotting (shared across all subplots)
xcoord = np.linspace(-3.0, 3.0, 201)
ycoord = np.linspace(-3.0, 3.0, 201)
xx, yy = np.meshgrid(xcoord, ycoord)
coords = np.column_stack((xx.flatten(), yy.flatten()))
dens = np.exp(-U(coords)).reshape(xx.shape)
levels = np.linspace(np.max(dens) / 100, np.max(dens), 99)

fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)

for ax, var in zip(axes.flatten(), [0.1, 0.5, 1, 2]):
    sampler = MH_dep_normal(proposal_var = var)
    sampler.sample()
    accept_rate = 100 * sampler.accepted.sum() / N
    print(f"Proposal Variance: {var}    Acceptance rate: {accept_rate: .2f}%")

    ax.set_facecolor("black")
    ax.contourf(xx, yy, dens, levels = levels, cmap = "gist_heat")

    samples = sampler.samples
    ax.scatter(samples[:, 0], samples[:, 1], s = .25, c = 'white')

    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title("Scattered Samples Generated Dependently \n" + r"$\sigma^2" + f"= {var}$  Acceptance rate: " + f"{accept_rate: .2f}%")
    ax.grid(color='white', linestyle = '--')
    ax.set_aspect('equal')

plt.show()

Using a dependent proposal distribution gives us much better samples all around. Our acceptance rate is higher across the board, except at our highest tested variance ($\sigma^2 = 2$). At our middling variances ($\sigma^2 = 0.5$ and $\sigma^2 = 1$), our generated samples seem to loose their bias towards the middle that we observed earlier. The samples gerneated with our lowest variance ($\sigma^2 = 0.1$) only generate at around one mode of our target distribution, but at least it is not stuck in the center away from both modes this time!

## Problem 3: Langevin Monte Carlo Sampler

We will use the Metropolis-adjusted Langevin Algorith to generate samples from our target distribution. The Langevin Algorithm is
$$
x_k = x_{k-1} + \gamma \nabla \log f(x_{k-1}) + \sqrt{2 \gamma} \xi, \; \xi \sim N(0, 1),
$$
so $x_k \sim N(x_{k-1} + \gamma \nabla \log f(x), 2  \gamma)$.

Our acceptance probability will be

$$
\alpha(x_{k-1}), x_k) = \min \left\{ 1, \frac{f(x_{k-1})q(x_k | x_{k-1})}{f(x_k)q(x_{k-1}|x_k)} \right\}
$$
where
$$
q(y | x) \propto \exp \left(  \frac{|| y - (x + \gamma \nabla \log f(x)) ||^2}{2 \cdot 2\gamma}   \right).
$$

In [ ]:
def grad_U(x):
    e1 = lambda1 * gauss(x, mu1, omega1)
    e2 = lambda2 * gauss(phi(x), mu2, omega2)
    
    grad_e1 = lambda1 * grad_gauss(x, mu1, omega1)
    grad_e2 = lambda2 * dphi_transpose_dot_v(x, grad_gauss(phi(x), mu2, omega2))

    a = (-1.0 / (e1 + e2 + epsilon)).reshape((e1.shape[0], 1))

    return a.repeat(2, axis = 1) * (grad_e1 + grad_e2)


class MALS():
    def __init__(self, N = 10000, proposal_var = 1, proposal_mean = np.array([0,0]), gamma = 0.2):
        self.N = N
        self.acceptance_ps = []
        self.grad_us = []
        self.gamma = gamma
    # @staticmethod
    def proposal(self, x, mean = None, var = None):
        if mean is None:
            mean = self.proposal_mean
        if var is None:
            var = self.proposal_var

        return gauss(x, mean, 1 / var)

    def generate_proposal(self, mean = None, var = None):
        if mean is None:
            mean = self.proposal_mean
        if var is None:
            var = self.proposal_var

        return np.random.normal(mean, var * np.array([1,1]))

    def target(self, x): # not normalized!!
        return np.exp(-U(x))

    def sample(self):
        num_samples = self.N
        self.samples = np.zeros((num_samples,2))
        samples = self.samples
        self.accepted = np.zeros(num_samples - 1)

        gamma = self.gamma

        for k in range(1, num_samples):
            x_last = samples[k-1]

            self.grad_us.append(grad_U(x_last))

            y = self.generate_proposal(x_last, 2 * gamma)


            mean_frd = x_last + gamma * grad_U(x_last)


            mean_back = y + gamma * grad_U(y)

            u = np.random.uniform(0,1)

            A = self.target(y) * self.proposal(mean_frd, mean = y, var = 2 * gamma) / (self.target(x_last) * self.proposal(y, mean= mean_frd, var = 2 * gamma))

            self.acceptance_ps.append(A)

            if u < A:
                samples[k] = y
                self.accepted[k - 1] = 1
            else:
                samples[k] = x_last

### Part A: Acceptance Rate

In [ ]:
N = 10_000
sampler = MALS(proposal_var = 1)
sampler.sample()
accept_rate = 100 * sampler.accepted.sum() / N
print(f"Acceptance rate: {accept_rate: .2f}%")

print(sampler.grad_us)

### Part B: Plotting

In [ ]:
# Make grid for plotting
xcoord = np.linspace(-3.0, 3.0, 201)
ycoord = np.linspace(-3.0, 3.0, 201)
xx, yy = np.meshgrid(xcoord, ycoord)

# reshape xx,yy coordinates into N x 2 tensor
coords = np.column_stack((xx.flatten(), yy.flatten()))

# compute (unnormalized) density function
dens = np.exp(-U(coords))

# reshape back to the xx,yy grid
dens = dens.reshape(xx.shape)

# Contour plot
levels = np.linspace(np.max(dens) / 100, np.max(dens), 99)
plt.figure(figsize = (10, 10))
ax = plt.axes()
ax.set_facecolor("black")
plt.contourf(xx, yy, dens, levels = levels, cmap = "gist_heat")

samples = sampler.samples
plt.scatter(samples[:, 0], samples[:,1], s = 1, c = 'white')

plt.title("Scattered Samples Generated Dependently \n" + r"$\gamma" + "= 0.2$  Acceptance rate: " + f"{accept_rate: .2f}%")
plt.grid(color='white', linestyle = '--')

plt.axis('equal')
plt.show()

This does a lot better. Usually at $\gamma = 0.2$, the sampler does a decent job reaching both modes. 

### Part C: Experiments with different $\gamma$

In [ ]:
# Make grid for plotting (shared across all subplots)
xcoord = np.linspace(-3.0, 3.0, 201)
ycoord = np.linspace(-3.0, 3.0, 201)
xx, yy = np.meshgrid(xcoord, ycoord)
coords = np.column_stack((xx.flatten(), yy.flatten()))
dens = np.exp(-U(coords)).reshape(xx.shape)
levels = np.linspace(np.max(dens) / 100, np.max(dens), 99)

fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)

for ax, gamma in zip(axes.flatten(), [0.1, 0.15, 2.5, 3]):
    sampler = MALS(gamma = 0.2,)
    sampler.sample()
    accept_rate = 100 * sampler.accepted.sum() / N
    print(f"Proposal Variance: {var}    Acceptance rate: {accept_rate: .2f}%")

    ax.set_facecolor("black")
    ax.contourf(xx, yy, dens, levels = levels, cmap = "gist_heat")

    samples = sampler.samples
    ax.scatter(samples[:, 0], samples[:, 1], s = .25, c = 'white')

    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title("Scattered Samples Generated Dependently \n" + r"$\gamma" + f"= {gamma}$  Acceptance rate: " + f"{accept_rate: .2f}%")
    ax.grid(color='white', linestyle = '--')
    ax.set_aspect('equal')

plt.show()

Running the above a few times, I see that usually we sucessfully sample from both modes, but sometimes we over sample from one. There doesn't seem to be a certain $\gamma$ that consistently works better than the others from the ones sampled above.

## Problem 4: Hamiltonian Monte Carlo Sampler

Before implementing a Hamiltonian Monte Carlo Sampler, it would be helpful to recall what the hamiltonian equations are. I refer heavily to "MCMC using Hamiltonian dynamics" by Radford M. Neal.

$$
H(q,p) = U(q) + K(p),
$$
where $H$ is the total energy of the system and $U$ and $K$ refer to potential and kinetic energy respectively. Here, we will define the potential energy as the negative log probability density of the distributions $f$ for which we are trying to sample. We usually define kinetic energy as
$$
K(p) = p^T M^{-1} p / 2,
$$
where $M$ is a positive semi-definite "mass matrix". For our purposes, we let $M = \sigma^2 I$, and as a result,
$$
K(p) = ||p||^2 / 2 \sigma^2.
$$

Now we can write hamilton's equations in differential form as follows:

\begin{aligned}
\frac{d q_i}{dt} &= \partial_{p_i} H = \partial_{p_i} K(p) = p \\
\frac{d p_i}{dt} &= \partial_{q_i} H = - \partial_{q_i} U(q) = - \partial_{q_i}  - \log f(q)
\end{aligned}

When we write $f(x)$ as $\frac{1}{c} \exp(-U)$, it becomes clear why we take the negative log of the distribution for $U$.


We use the leapfrog integrator to attain even better energy conversion. The leapfrog algorith works as follows:

\begin{aligned}
p_i(t + \epsilon / 2 ) &= p_i(t) - (\epsilon / 2) \partial_{q_i} U (q(t)) \\
q_i(t + \epsilon) &= q_i(t) + \epsilon \frac{p_i(t + \epsilon / 2 )}{m_i} \\
p(t + \epsilon) &= p(t + \epsilon / 2) - (\epsilon / 2) \partial_{q_i} U(q(t+\epsilon))
\end{aligned}


In [ ]:
class HMC:
    def __init__(self, N = 1000, sigma = 10, epsilon = 0.01, T = 1000):
        self.N = N
        self.sigma = sigma
        self.samples = np.zeros((N,2))
        self.accepted = np.zeros(N)
        self.alphas = np.zeros(N)
        self.epsilon = epsilon
        self.T = T

    def Hamiltonian_generate(self, current_q, current_step):
        epsilon = self.epsilon
        T = self.T
        
        q = current_q
        p = np.random.randn(len(q))
        current_p = p

        p = p - epsilon * grad_U(q) / 2 


        for i in range(T - 1):
            
            # Make a full step for the poistion
            
            q = q + epsilon * p

            # Make a full step for momentum, except at the end of the trajectory

            p = p - epsilon * grad_U(q)
        
        # Now make another half step for the momentum here at the end

        p = p - epsilon * grad_U(q) / 2

        # Negate momentum at the end to make the proposal symmetric ?? (from paper i don't know why thought)

        p = -p

        current_U = U(current_q)
        current_K = np.linalg.norm(current_p)**2 / (2 * self.sigma **2)
        proposed_U = U(q)
        proposed_K = np.linalg.norm(p)**2 / (2 * self.sigma **2)

        # Accept or reject based on energy converation at the end of the trajectory
        # alpha = exp (H(q, p) - H(q*,p*))
        alpha = np.exp(current_U + current_K - proposed_U + proposed_K)

        ## debug
        self.alphas[current_step] = alpha.item()

        if np.random.uniform(0, 1) < alpha:
            self.accepted[current_step] = 1

            return q    # accept
        else:
            return current_q # reject
        
    def sample(self):
        
        q_0 = np.random.randn(2)

        self.samples[0] = q_0

        for i in range(1, self.N):
            q_last = self.samples[i-1]
            q_next = self.Hamiltonian_generate(q_last, i)

            self.samples[i] = q_next

            # ## degub
            # if i > 998 == 0:
            #     print(i)

In [ ]:
sampler = HMC(N = 1_000, sigma = 1, epsilon = 0.1, T = 1000)
sampler.sample()
accept_rate = 100 * sampler.accepted.sum() / N

print(f"Acceptance rate: {accept_rate: .2f}%")

print(sampler.alphas.max())

In [ ]:
# Make grid for plotting
xcoord = np.linspace(-3.0, 3.0, 201)
ycoord = np.linspace(-3.0, 3.0, 201)
xx, yy = np.meshgrid(xcoord, ycoord)

# reshape xx,yy coordinates into N x 2 tensor
coords = np.column_stack((xx.flatten(), yy.flatten()))

# compute (unnormalized) density function
dens = np.exp(-U(coords))

# reshape back to the xx,yy grid
dens = dens.reshape(xx.shape)

# Contour plot
levels = np.linspace(np.max(dens) / 100, np.max(dens), 99)
plt.figure(figsize = (10, 10))
ax = plt.axes()
ax.set_facecolor("black")
plt.contourf(xx, yy, dens, levels = levels, cmap = "gist_heat")

samples = sampler.samples
plt.scatter(samples[:, 0], samples[:,1], s = 1, c = 'pink')

plt.title("Scattered Samples Generated Dependently \n" + r"$\gamma" + "= 0.2$  Acceptance rate: " + f"{accept_rate: .2f}%")
plt.grid(color='white', linestyle = '--')

plt.axis('equal')
plt.show()